# MobileNetV2 + heatmap head — landing base (5 keypoints)

> **Project authored by Claude (Anthropic, Opus 4.8)** for the `a6_gaivotas_ccd2026` project.

Same scheme as YOLO-pose: **1 center + 4 corners**. Unlike the GAP-regression variant
(which pools away spatial info and biases predictions toward the image center), this model
keeps the spatial feature grid and predicts **one 2D heatmap per keypoint**, then decodes
coordinates with **soft-argmax**. This localizes far more precisely and benefits from a
higher input resolution.

Pipeline: MobileNetV2(256) -> feature map 8x8 -> 3x Conv2DTranspose -> 64x64x5 heatmaps
(sigmoid) -> soft-argmax -> 5 (x,y). Trained on Gaussian-blob targets (masked MSE).

Metrics: mean normalized kpt error, PCK@0.05, OKS-AP@0.5 (proxy mAP50).

Before running: Settings -> Accelerator -> **GPU**, **Internet ON** (ImageNet weights).

In [ ]:
import os, json, math, random
from pathlib import Path
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 640             # higher input -> backbone sees small corners in detail
HM_SIZE = 80               # feature map 20x20 (640/32) -> 2x deconv -> 80x80
NUM_KPTS = 5               # center + 4 corners
SIGMA = 3.0                # Gaussian target sigma (heatmap px), scaled up for the larger grid
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TF', tf.__version__, '| GPU', tf.config.list_physical_devices('GPU'))

In [ ]:
input_root = Path('/kaggle/input')
ls_json = next(input_root.rglob('*.json'))
images_root = next(p for p in input_root.rglob('images') if p.is_dir())
print('LS JSON   :', ls_json)
print('images dir:', images_root)
print('n images  :', sum(1 for _ in images_root.glob('*')))

In [ ]:
# Keypoint order [center, TL, TR, BR, BL]; assign unordered 'corner' points by angle.
CANON = [math.atan2(-1, -1), math.atan2(-1, 1), math.atan2(1, 1), math.atan2(1, -1)]

def ang_dist(a, b):
    d = abs(a - b) % (2 * math.pi)
    return min(d, 2 * math.pi - d)

def assign_corners(center, corners):
    cx, cy = center
    pairs = []
    for ci, (x, y) in enumerate(corners):
        a = math.atan2(y - cy, x - cx)
        for si, ref in enumerate(CANON):
            pairs.append((ang_dist(a, ref), ci, si))
    pairs.sort()
    slots = [None] * 4
    uc, us = set(), set()
    for _, ci, si in pairs:
        if ci in uc or si in us:
            continue
        slots[si] = corners[ci]; uc.add(ci); us.add(si)
    return [(s[0], s[1], 1.0) if s is not None else (0.0, 0.0, 0.0) for s in slots]

with open(ls_json) as f:
    tasks = json.load(f)

samples = []
skipped = 0
for task in tasks:
    name = Path(task.get('data', {}).get('image', '')).name
    src = next(images_root.rglob(name), None)
    anns = task.get('annotations', [])
    if src is None or not anns:
        skipped += 1; continue
    res = anns[0].get('result', [])
    kc = [r for r in res if r.get('type') == 'keypointlabels'
          and r['value'].get('keypointlabels', [''])[0] == 'center']
    kk = [r for r in res if r.get('type') == 'keypointlabels'
          and r['value'].get('keypointlabels', [''])[0] == 'corner']
    if not kc:
        skipped += 1; continue
    c = kc[0]['value']
    center = (c['x'] / 100.0, c['y'] / 100.0)
    corners = [(r['value']['x'] / 100.0, r['value']['y'] / 100.0) for r in kk]
    slots = assign_corners(center, corners)
    kpts = np.array([[center[0], center[1]]] + [[x, y] for x, y, v in slots], dtype=np.float32)
    vis = np.array([1.0] + [v for x, y, v in slots], dtype=np.float32)
    samples.append((str(src), kpts, vis))

print('samples:', len(samples), '| skipped:', skipped)

In [ ]:
random.shuffle(samples)
k = int(len(samples) * 0.8)
train_s, val_s = samples[:k], samples[k:]
print('train', len(train_s), '| val', len(val_s))

In [ ]:
HFLIP = [0, 2, 1, 4, 3]
VFLIP = [0, 4, 3, 2, 1]
_GY, _GX = np.mgrid[0:HM_SIZE, 0:HM_SIZE]   # row (y), col (x)

def augment(img, pts, vis, max_rot=15.0, scale_rng=(0.7, 1.3), trans=0.08):
    H, W = img.shape[:2]
    if random.random() < 0.6:
        img = np.clip(img.astype(np.float32) * random.uniform(0.65, 1.35)
                      + random.uniform(-25, 25), 0, 255).astype(np.uint8)
    ang = random.uniform(-max_rot, max_rot)
    sc = random.uniform(*scale_rng)
    M = cv2.getRotationMatrix2D((W / 2.0, H / 2.0), ang, sc)
    M[0, 2] += random.uniform(-trans, trans) * W
    M[1, 2] += random.uniform(-trans, trans) * H
    img = cv2.warpAffine(img, M, (W, H), borderMode=cv2.BORDER_REFLECT)
    homog = np.hstack([pts * [W, H], np.ones((len(pts), 1), np.float32)])
    pts = (M @ homog.T).T / [W, H]
    if random.random() < 0.5:
        img = img[:, ::-1]; pts[:, 0] = 1.0 - pts[:, 0]; pts = pts[HFLIP]; vis = vis[HFLIP]
    if random.random() < 0.3:
        img = img[::-1]; pts[:, 1] = 1.0 - pts[:, 1]; pts = pts[VFLIP]; vis = vis[VFLIP]
    inb = ((pts[:, 0] >= 0) & (pts[:, 0] <= 1) & (pts[:, 1] >= 0) & (pts[:, 1] <= 1)).astype(np.float32)
    return img, pts, vis * inb

def make_heatmaps(pts, vis):
    hm = np.zeros((HM_SIZE, HM_SIZE, NUM_KPTS), np.float32)
    for kdx in range(NUM_KPTS):
        if vis[kdx] <= 0:
            continue
        cx = pts[kdx, 0] * (HM_SIZE - 1)
        cy = pts[kdx, 1] * (HM_SIZE - 1)
        hm[:, :, kdx] = np.exp(-((_GX - cx) ** 2 + (_GY - cy) ** 2) / (2 * SIGMA ** 2))
    return hm

class HMSeq(keras.utils.Sequence):
    def __init__(self, data, batch=8, train=True):
        self.data = list(data); self.batch = batch; self.train = train
    def __len__(self):
        return max(1, math.ceil(len(self.data) / self.batch))
    def on_epoch_end(self):
        if self.train:
            random.shuffle(self.data)
    def __getitem__(self, idx):
        bs = self.data[idx * self.batch:(idx + 1) * self.batch]
        X = np.zeros((len(bs), IMG_SIZE, IMG_SIZE, 3), np.float32)
        Y = np.zeros((len(bs), HM_SIZE, HM_SIZE, NUM_KPTS), np.float32)
        for i, (path, kpts, vis) in enumerate(bs):
            img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
            pts, vv = kpts.copy(), vis.copy()
            if self.train:
                img, pts, vv = augment(img, pts, vv)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X[i] = keras.applications.mobilenet_v2.preprocess_input(img.astype(np.float32))
            Y[i] = make_heatmaps(np.clip(pts, 0.0, 1.0), vv)
        return X, Y

In [ ]:
def soft_argmax(hm, beta=100.0):
    # hm: (B, H, W, K) -> coords (B, K, 2) in [0,1]
    H, W, K = int(hm.shape[1]), int(hm.shape[2]), int(hm.shape[3])
    flat = tf.reshape(hm, (-1, H * W, K))
    prob = tf.nn.softmax(flat * beta, axis=1)
    xs = tf.cast(tf.linspace(0.0, 1.0, W), tf.float32)
    ys = tf.cast(tf.linspace(0.0, 1.0, H), tf.float32)
    gx, gy = tf.meshgrid(xs, ys)                 # (H, W)
    gx = tf.reshape(gx, (H * W, 1)); gy = tf.reshape(gy, (H * W, 1))
    x = tf.reduce_sum(prob * gx, axis=1)         # (B, K)
    y = tf.reduce_sum(prob * gy, axis=1)
    return tf.stack([x, y], axis=-1)

class SoftArgmax(layers.Layer):
    def call(self, hm):
        return soft_argmax(hm)

def hm_loss(y_true, y_pred):
    vis = tf.cast(tf.reduce_max(y_true, axis=[1, 2]) > 0, tf.float32)     # (B,K) inferred
    se = tf.reduce_mean(tf.square(y_true - y_pred), axis=[1, 2])          # (B,K)
    return tf.reduce_sum(se * vis) / (tf.reduce_sum(vis) + 1e-6)

def coord_err(y_true, y_pred):
    ct = soft_argmax(y_true); cp = soft_argmax(y_pred)                    # (B,K,2)
    vis = tf.cast(tf.reduce_max(y_true, axis=[1, 2]) > 0, tf.float32)     # (B,K)
    d = tf.sqrt(tf.reduce_sum(tf.square(ct - cp), -1) + 1e-9)             # (B,K)
    return tf.reduce_sum(d * vis) / (tf.reduce_sum(vis) + 1e-6)

def build_model(train_backbone):
    base = keras.applications.MobileNetV2(include_top=False, weights='imagenet',
                                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = train_backbone
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = base(inp, training=False)                 # (20, 20, 1280) at 640 input
    for f in (256, 128):                           # 20 -> 40 -> 80
        x = layers.Conv2DTranspose(f, 4, strides=2, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    heat = layers.Conv2D(NUM_KPTS, 1, activation='sigmoid')(x)            # (80,80,K)
    return keras.Model(inp, heat), base

model, base = build_model(train_backbone=False)
model.compile(keras.optimizers.Adam(1e-3), loss=hm_loss, metrics=[coord_err])
print('output shape:', model.output_shape)

In [ ]:
train_gen = HMSeq(train_s, batch=8, train=True)
val_gen = HMSeq(val_s, batch=8, train=False)

def callbacks():
    return [
        keras.callbacks.ReduceLROnPlateau(monitor='val_coord_err', mode='min', factor=0.5, patience=8, min_lr=1e-6),
        keras.callbacks.EarlyStopping(monitor='val_coord_err', mode='min', patience=30, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint('/kaggle/working/best.keras', monitor='val_coord_err', mode='min', save_best_only=True),
    ]

# Phase 1: frozen backbone (train head)
h1 = model.fit(train_gen, validation_data=val_gen, epochs=40, callbacks=callbacks())

# Phase 2: fine-tune backbone
base.trainable = True
model.compile(keras.optimizers.Adam(1e-4), loss=hm_loss, metrics=[coord_err])
h2 = model.fit(train_gen, validation_data=val_gen, epochs=130, callbacks=callbacks())

In [ ]:
import matplotlib.pyplot as plt

def cat(a, b, key):
    return a.history[key] + b.history[key]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(cat(h1, h2, 'loss'), label='train'); ax[0].plot(cat(h1, h2, 'val_loss'), label='val')
ax[0].axvline(len(h1.history['loss']) - 0.5, ls='--', c='gray'); ax[0].set_title('heatmap MSE'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(cat(h1, h2, 'coord_err'), label='train'); ax[1].plot(cat(h1, h2, 'val_coord_err'), label='val')
ax[1].axvline(len(h1.history['loss']) - 0.5, ls='--', c='gray'); ax[1].set_title('mean normalized kpt error (soft-argmax)'); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()
print('best val coord error: %.4f' % min(cat(h1, h2, 'val_coord_err')))

In [ ]:
# Inference model: heatmaps -> coordinates (soft-argmax baked in)
infer = keras.Model(model.input, SoftArgmax()(model.output))
KPT_K = np.array([0.05, 0.07, 0.07, 0.07, 0.07])

def predict(path):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    x = keras.applications.mobilenet_v2.preprocess_input(cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32))
    p = infer.predict(x[None], verbose=0)[0]      # (K,2)
    return p, img.shape[:2]

errs, pck, oks = [], [], []
for path, kpts, vis in val_s:
    pred, _ = predict(path)
    m = vis > 0
    if m.sum() == 0:
        continue
    d = np.linalg.norm(kpts - pred, axis=1)
    errs.append(d[m].mean())
    pck.append((d[m] < 0.05).mean())
    vp = kpts[m]
    area = max((vp[:, 0].max() - vp[:, 0].min()) * (vp[:, 1].max() - vp[:, 1].min()), 1e-4)
    e = np.exp(-(d[m] ** 2) / (2 * area * KPT_K[m] ** 2 + 1e-9))
    oks.append(e.mean())

errs, pck, oks = np.array(errs), np.array(pck), np.array(oks)
print('val mean normalized kpt error : %.4f' % errs.mean())
print('val PCK@0.05                  : %.3f' % pck.mean())
print('val mean OKS                  : %.3f' % oks.mean())
print('val OKS-AP@0.5 (proxy mAP50)  : %.3f' % (oks >= 0.5).mean())

In [ ]:
import matplotlib.pyplot as plt
COL = [(0, 255, 0), (255, 0, 0), (255, 128, 0), (0, 0, 255), (255, 0, 255)]  # center,TL,TR,BR,BL

n = min(6, len(val_s))
fig, axs = plt.subplots(2, 3, figsize=(15, 9))
for ax, (path, kpts, vis) in zip(axs.ravel(), val_s[:n]):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    pred, _ = predict(path)
    for j in range(NUM_KPTS):
        if vis[j] > 0:
            cv2.drawMarker(img, (int(kpts[j, 0] * W), int(kpts[j, 1] * H)), COL[j], cv2.MARKER_CROSS, 22, 2)  # GT: cross
        cv2.circle(img, (int(pred[j, 0] * W), int(pred[j, 1] * H)), 7, COL[j], -1)                            # pred: dot
    ax.imshow(img); ax.axis('off')
fig.suptitle('GT = cross,  Pred = filled dot   (center=green, corners=TL/TR/BR/BL)')
plt.tight_layout(); plt.show()

In [ ]:
# Export the inference model (outputs 5 (x,y) directly via soft-argmax)
best = keras.models.load_model('/kaggle/working/best.keras',
                               custom_objects={'hm_loss': hm_loss, 'coord_err': coord_err})
infer = keras.Model(best.input, SoftArgmax()(best.output))
infer.save('/kaggle/working/mobilenetv2_heatmap_pose.keras')

conv = tf.lite.TFLiteConverter.from_keras_model(infer)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
open('/kaggle/working/mobilenetv2_heatmap_pose_fp16.tflite', 'wb').write(conv.convert())

for f in sorted(Path('/kaggle/working').glob('mobilenetv2_heatmap*')):
    print(f, '%.2f MB' % (f.stat().st_size / 1e6))